# Candidate Discovery

This notebook is the readable companion to `2_data/scripts/candidate_discovery.py`.

The goal is not to select the final model variables. The goal is to make the target and predictor search space auditable before modeling starts.

## Research Motivation

A strong empirical project should be able to explain why variables were considered, screened, included, deferred, or dropped. This notebook shows the candidate discovery outputs that support that argument.

## Optional Google Colab Setup

Run the next cell only when opening this notebook in Google Colab. In local Jupyter, the cell detects that it is not in Colab and safely skips itself.

In [ ]:
# Optional: run this cell only when using Google Colab.
import importlib.util
import os
import subprocess
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    REPO_URL = "https://github.com/songjie1025/env-innovation-prediction.git"
    REPO_DIR = Path("/content/env-innovation-prediction")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run(["python", "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Local environment detected; skipping Colab setup.")

In [ ]:
from pathlib import Path
import sys

from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "2_data" / "scripts").exists():
            return candidate
    for child in start.glob("*/README.md"):
        candidate = child.parent
        if (candidate / "2_data" / "scripts").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root containing README.md and 2_data/scripts.")

ROOT = find_project_root(Path.cwd().resolve())

SCRIPT_DIR = ROOT / "2_data" / "scripts"
PROCESSED_DIR = ROOT / "2_data" / "processed"
sys.path.append(str(SCRIPT_DIR))

from candidate_discovery import run_candidate_discovery

## Load Or Regenerate Discovery Outputs

By default, the notebook reads committed processed outputs. Set `RUN_DISCOVERY = True` to refresh the catalogs from local metadata and optional lightweight metadata scans.

In [ ]:
RUN_DISCOVERY = False
SCAN_WORLD_BANK_METADATA = False

required_outputs = {
    "targets": PROCESSED_DIR / "target_candidate_catalog.csv",
    "predictors": PROCESSED_DIR / "predictor_candidate_catalog.csv",
    "summary": PROCESSED_DIR / "candidate_discovery_summary.md",
}

if RUN_DISCOVERY or not all(path.exists() for path in required_outputs.values()):
    target_catalog, predictor_catalog = run_candidate_discovery(
        scan_world_bank=SCAN_WORLD_BANK_METADATA,
        output_dir=PROCESSED_DIR,
    )
else:
    target_catalog = pd.read_csv(required_outputs["targets"])
    predictor_catalog = pd.read_csv(required_outputs["predictors"])

target_catalog.shape, predictor_catalog.shape

## Target Candidate Catalog

The table below documents candidate OECD patent source-variable combinations and a conservative reviewer-facing classification.

In [ ]:
target_columns = [
    "source_variable",
    "unit_measure_label",
    "type_label",
    "technology_domain_label",
    "regional_patent_office_label",
    "candidate_role",
    "include",
    "coverage_checked",
    "countries_with_data",
    "first_year",
    "last_year",
    "reason",
]

display(
    target_catalog.loc[:, [column for column in target_columns if column in target_catalog.columns]]
    .sort_values(["include", "candidate_role", "source_variable"], ascending=[False, True, True])
    .head(30)
)

In [ ]:
role_counts = target_catalog["candidate_role"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(role_counts.index, role_counts.values, color="#3b6ea8")
ax.set_title("OECD patent target candidate roles")
ax.set_xlabel("Candidate combinations")
ax.set_ylabel("")
plt.tight_layout()

## Predictor Candidate Catalog

The predictor catalog combines literature-driven concepts with database metadata screening and first-pass coverage status.

In [ ]:
predictor_columns = [
    "variable_concept",
    "candidate_indicator",
    "source",
    "source_variable",
    "literature_support",
    "coverage_checked",
    "coverage_summary",
    "already_downloaded",
    "include_decision",
    "decision_reason",
    "measurement_caveat",
    "lag_recommendation",
]

display(predictor_catalog.loc[:, [column for column in predictor_columns if column in predictor_catalog.columns]])

In [ ]:
decision_counts = predictor_catalog["include_decision"].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(decision_counts.index, decision_counts.values, color="#6c8f3d")
ax.set_title("Predictor candidate inclusion decisions")
ax.set_xlabel("Candidate indicators")
ax.set_ylabel("")
plt.tight_layout()

## Generated Summary

The markdown summary is written by the script so the discovery step can be reviewed without opening this notebook.

In [ ]:
summary_path = PROCESSED_DIR / "candidate_discovery_summary.md"
if summary_path.exists():
    display(Markdown(summary_path.read_text(encoding="utf-8")))
else:
    print("Run candidate discovery to generate the markdown summary.")